# Pipeline walkthrough

What the pipeline does to the real match, one stage at a time, ending with the submission file.
All the logic lives in `src/`, and this notebook only calls it and shows the result, so it can't
drift away from `run.py`.

**The outputs here contain match data.** Keep them out of cloud AI. `nbstripout` strips them from
commits.

---

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os, sys
sys.path[:0] = [os.path.abspath(p) for p in ('..', '../src', '../..', '../../starter', '../../evaluation')]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ball, dead_ball, features, model, parse, report, run, stage2, states, tags
from games import GAME, half_video_blocks

blocks = half_video_blocks(GAME)
blocks

## 1. Load the tracking

`parse.load()` reads the cache, or builds it from the raw file the first time. Everything is
indexed by video frame, in metres.

In [ ]:
data = parse.load()
{name: value.shape for name, value in data.items()}

## 2. Clean the ball

Impossible jumps are removed and gaps up to 1 second are bridged. Longer gaps stay empty, on
purpose: a long gap is itself a clue that play may have stopped.

In [ ]:
raw = ~np.isnan(data['ball_xy'][:, 0])
clean_ball = ball.clean(data['ball_xy'])
cleaned = ~np.isnan(clean_ball[:, 0])
graded = np.zeros(len(raw), dtype=bool)
for _, first, last in blocks:
    graded[first:last + 1] = True

print('ball seen, raw      %.1f%% of graded frames' % (100 * raw[graded].mean()))
print('ball seen, cleaned  %.1f%% of graded frames' % (100 * cleaned[graded].mean()))

per_minute = lambda seen: seen[:len(seen) // 1500 * 1500].reshape(-1, 1500).mean(axis=1) * 100
fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(per_minute(raw), label='raw')
ax.plot(per_minute(cleaned), label='cleaned')
ax.set_xlabel('minute of video'); ax.set_ylabel('% frames with a ball'); ax.legend()
ax.set_title('Ball coverage, before and after cleaning')
plt.show()

## 3. Dead-ball signals

Players slow down at a stoppage. The grey shading shows what the hand-written rules call dead over
five minutes of play, and the rules feed the model as one clue among many.

In [ ]:
speed = dead_ball.team_speed(data['player_speed'])
rule_dead = dead_ball.detect(data, clean_ball)

start = blocks[0][1]
stop = start + 5 * 60 * 25
frames = np.arange(start, stop)
fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(frames, speed[start:stop], lw=.8)
ax.axhline(dead_ball.SLOW_SPEED, color='k', ls='--', lw=.8, label='slow limit')
ax.fill_between(frames, 0, speed[start:stop].max(), where=rule_dead[start:stop], color='grey',
                alpha=.3, label='rules: dead')
ax.set_xlabel('video frame'); ax.set_ylabel('mean player speed, m/s'); ax.legend()
plt.show()

## 4. Clues and model scores

`features.build` makes one row of clues per frame. The model turns them into a score for H, A and
D. These scores are still jumpy, which is why the next stage exists.

In [ ]:
booster = model.load()
names, X = features.build(data, blocks)
scores = booster.predict(X)
print('%d clues, %d frames' % (len(names), len(X)))

fig, ax = plt.subplots(figsize=(13, 3.5))
for index, (label, colour) in enumerate(zip('HAD', ('#a50044', '#ee2523', '#888888'))):
    ax.plot(frames, scores[start:stop, index], color=colour, lw=.8, label=label)
ax.set_xlabel('video frame'); ax.set_ylabel('model score'); ax.legend()
ax.set_title('Per-frame scores, first five minutes')
plt.show()

### Which clues matter most

If this list is led by something fragile, the model may not transfer well to the real tracking.

In [ ]:
importance = pd.Series(booster.feature_importance('gain'), index=booster.feature_name())
share = (importance / importance.sum()).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 4.5))
share[::-1].plot.barh(ax=ax)
ax.set_xlabel("share of the model's total gain")
plt.show()

## 5. Clean possessions

A second model (`src/stage2.py`) first rereads these scores in context. Then the best-path
step picks one sequence for each half and pays `CHANGE_COST` for every change, which
is what removes the flicker. Compare the possession count before and after.

In [ ]:
before = np.full(len(scores), 'X', dtype='<U1')
for _, first, last in blocks:
    before[first:last + 1] = states.STATES[scores[first:last + 1].argmax(axis=1)]
labels = run.predict(data, blocks, booster, stage2.load())

pd.DataFrame({'per-frame only': report.overview(before, blocks),
              'after best path': report.overview(labels, blocks)})

In [ ]:
report.plot_timeline(labels, blocks)
plt.show()

In [ ]:
spells = report.possessions(labels, blocks)
fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(spells['seconds'].clip(upper=90), bins=45)
ax.set_xlabel('possession length, seconds (90+ grouped)'); ax.set_ylabel('possessions')
plt.show()

## 6. Check against Smart Tagging

The tags are automatic, so this is allowed on this match. They are phases, not possessions, so full
agreement is not the goal. The rules version is shown for reference.

In [ ]:
pd.DataFrame({'rules': tags.agreement(run.predict_rules(data, blocks)),
              'current': tags.agreement(labels)}).round(3)

## 7. macroF1 and spellF1 on the public 5 minutes

These are real scores, but on only 17 possessions, so **this is a check that the pipeline works, not
a result.** One missed possession moves `spellF1` by about six points, and the range shows how wide
that noise is. Never choose between two versions here, because the rules forbid it. Use
`score_external.py` or notebook 05 for that.

Your full-match score stays private until the results.

In [ ]:
import possession_metrics as pm

gt = pm.label_codes(pm.read_label_csv('../../evaluation/gt/BAR-ATH_possession_gt_public.csv'))
baseline = pm.read_label_csv('../../evaluation/results/baseline_smooth50.csv')['label'].values


def public_score(labels):
    result = pm.evaluate(gt, pm.label_codes(labels), bootstrap_n=1000)
    return {'spellF1': round(result['spell_f1_mean'], 4),
            'spellF1 range': '%.2f .. %.2f' % (result['spell_bootstrap_low'],
                                               result['spell_bootstrap_high']),
            'macroF1': round(result['macro_f1'], 4),
            'possessions': result['n_pred_spells'],
            'true possessions': result['n_gt_spells']}


pd.DataFrame({'current': public_score(labels), 'baseline_smooth50': public_score(baseline)})

## 8. Write and validate the file

The same file `python victor/run.py` writes.

In [ ]:
path = run.write_submission('../../evaluation/results/victor.csv', labels)
!"{sys.executable}" ../../evaluation/validate_submission.py "{path}"